# 03 — Importación y deduplicación de fuentes

Este notebook incorpora exportaciones bibliográficas o institucionales al corpus de evidencia. El flujo es:

1. registrar cada archivo exportado;
2. adaptar sus columnas a un esquema común;
3. normalizar DOI, títulos y años;
4. detectar duplicados exactos;
5. preservar versiones distintas de instrumentos oficiales;
6. generar candidatos para el registro maestro.

La deduplicación automática no reemplaza la revisión humana. Los casos removidos quedan documentados en un archivo de auditoría.


In [ ]:
from pathlib import Path
import shutil
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from evidence_review.importing import (
    deduplicate_sources,
    load_import_mappings,
    read_tabular_export,
    standardise_export,
    to_source_registry,
)

RAW_EXPORTS = ROOT / "data" / "raw" / "search_exports"
INTERIM = ROOT / "data" / "interim"
RAW_EXPORTS.mkdir(parents=True, exist_ok=True)
INTERIM.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Place search exports in: {RAW_EXPORTS.relative_to(ROOT)}")


## 1. Cargar las equivalencias de columnas

Las plataformas usan nombres de columnas distintos. El archivo `config/import_mappings.yml` define equivalencias para exportaciones genéricas, Scopus, Web of Science, OpenAlex y Crossref.


In [ ]:
mapping_path = ROOT / "config" / "import_mappings.yml"
mapping_config = load_import_mappings(mapping_path)

supported_platforms = sorted(mapping_config["platforms"])
print("Supported platforms:", ", ".join(supported_platforms))


## 2. Crear el manifiesto de importación

Cada archivo colocado en `data/raw/search_exports/` debe aparecer en el manifiesto. El campo `platform` debe coincidir con una plataforma definida en el YAML.


In [ ]:
manifest_template = ROOT / "data" / "templates" / "import_manifest.csv"
manifest_path = INTERIM / "import_manifest_working.csv"

if not manifest_path.exists():
    shutil.copyfile(manifest_template, manifest_path)
    print(f"Created: {manifest_path.relative_to(ROOT)}")
else:
    print(f"Already exists: {manifest_path.relative_to(ROOT)}")

manifest = pd.read_csv(manifest_path)
manifest


Ejemplo de fila del manifiesto:

| file_name | platform | source_type | source_status | source_language |
|---|---|---|---|---|
| `openalex_pilot_en.csv` | `openalex` | `scientific_article` | `peer_reviewed` | `en` |

No agregues al manifiesto archivos que todavía no existan en `data/raw/search_exports/`.


## 3. Revisar los archivos disponibles


In [ ]:
available_files = sorted(
    path.name
    for path in RAW_EXPORTS.iterdir()
    if path.is_file() and path.suffix.lower() in {".csv", ".tsv", ".txt"}
)

print(f"Tabular exports found: {len(available_files)}")
available_files


## 4. Importar y estandarizar

El notebook procesa solo las filas del manifiesto que tengan un archivo existente. Si todavía no hay exportaciones, muestra instrucciones y continúa sin error.


In [ ]:
standardised_batches = []
import_errors = []

for row in manifest.fillna("").to_dict(orient="records"):
    file_name = str(row.get("file_name", "")).strip()
    platform = str(row.get("platform", "")).strip()

    if not file_name:
        continue

    export_path = RAW_EXPORTS / file_name
    if not export_path.exists():
        import_errors.append({
            "file_name": file_name,
            "error": "file_not_found",
        })
        continue

    if platform not in mapping_config["platforms"]:
        import_errors.append({
            "file_name": file_name,
            "error": f"unknown_platform:{platform}",
        })
        continue

    try:
        raw = read_tabular_export(export_path)
        defaults = {
            "source_type": row.get("source_type", ""),
            "source_status": row.get("source_status", ""),
            "source_language": row.get("source_language", ""),
        }
        standard = standardise_export(
            raw,
            mapping_config["platforms"][platform],
            defaults=defaults,
            import_file=file_name,
            import_platform=platform,
        )
        standardised_batches.append(standard)
    except Exception as exc:
        import_errors.append({
            "file_name": file_name,
            "error": f"{type(exc).__name__}: {exc}",
        })

if standardised_batches:
    imported = pd.concat(standardised_batches, ignore_index=True, sort=False)
else:
    imported = pd.DataFrame()

errors_df = pd.DataFrame(import_errors)

print(f"Imported records: {len(imported)}")
print(f"Import errors: {len(errors_df)}")
display(errors_df)


## 5. Deduplicar y mantener una auditoría

Prioridad de deduplicación:

1. DOI normalizado;
2. título normalizado + año;
3. título + año + fecha de versión para instrumentos oficiales.

Dentro de un grupo duplicado se retiene el registro con mayor completitud de metadatos. Todos los miembros del grupo quedan en el archivo de auditoría.


In [ ]:
if imported.empty:
    retained = imported.copy()
    duplicate_audit = pd.DataFrame()
    print("No records available for deduplication.")
else:
    retained, duplicate_audit = deduplicate_sources(imported)
    print(f"Records before deduplication: {len(imported)}")
    print(f"Records retained: {len(retained)}")
    print(f"Duplicate records audited: {len(duplicate_audit)}")

display(duplicate_audit.head(20))


## 6. Generar candidatos para el registro maestro


In [ ]:
if retained.empty:
    registry_candidates = pd.DataFrame()
else:
    registry_candidates = to_source_registry(retained)

outputs = {
    "imported_sources_raw.csv": imported,
    "deduplicated_sources.csv": retained,
    "duplicate_audit.csv": duplicate_audit,
    "source_registry_candidates.csv": registry_candidates,
    "import_errors.csv": errors_df,
}

for file_name, frame in outputs.items():
    output_path = INTERIM / file_name
    frame.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(frame):>5} rows -> {output_path.relative_to(ROOT)}")


## 7. Validaciones mínimas antes del cribado


In [ ]:
checks = {
    "imported_records": len(imported),
    "retained_records": len(retained),
    "duplicate_audit_rows": len(duplicate_audit),
    "missing_titles_retained": (
        int(retained["title"].fillna("").astype(str).str.strip().eq("").sum())
        if "title" in retained else 0
    ),
    "duplicate_source_ids": (
        int(registry_candidates["source_id"].duplicated().sum())
        if "source_id" in registry_candidates else 0
    ),
    "import_errors": len(errors_df),
}

pd.Series(checks, name="value").to_frame()


## Criterio para avanzar

Antes de construir el corpus de cribado:

- todas las exportaciones deben estar registradas en el manifiesto;
- `import_errors.csv` debe revisarse;
- no deben existir títulos vacíos entre los registros retenidos;
- los identificadores `source_id` deben ser únicos;
- `duplicate_audit.csv` debe revisarse manualmente;
- las distintas versiones de normas, planes o estrategias no deben eliminarse por error.

El siguiente notebook será `04_parse_and_segment_documents.ipynb`, pero solo después de validar un corpus piloto deduplicado.
